# 中证800 V85：V46 市场状态条件化与 Placebo 实验

本 notebook 固定 V46 的原始 `alpha_1m`、LightGBM 参数、120 轮、训练折叠和 `top8_board_cap`，只检验市场状态是否能稳定改善个股因子的条件关系：

- `A0_full_v46`：精确 V46 基线，不含市场状态
- `M1_true_state`：`full_v46 +` 点时真实市场状态
- `P1_placebo_state`：同样数量的状态特征，但训练月随机错位、测试月只抽取训练期状态

状态数据由聚宽 API 按 `feature_date` 构建并缓存。评估覆盖 AUC、PR-AUC、Precision、Recall、MAP、NDCG、RankIC、Top8 收益、年度 fold、弱窗口/强窗口、持仓重合、状态重要性和 moving-block bootstrap。M1 必须同时战胜 A0 和 P1 才能进入回测候选。


## 0. 导入、进度条与绘图基础


In [ ]:
import os
import gc
import math
import warnings
import time
import builtins as _bi
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from jqdata import get_price, get_index_stocks
except Exception:
    get_price = None
    get_index_stocks = None

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 260)
pd.set_option("display.width", 260)
pd.set_option("display.max_rows", 160)

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


def progress_iter(iterable, total=None, desc="progress", leave=True):
    if tqdm is not None:
        return tqdm(iterable, total=total, desc=desc, leave=leave)
    def _gen():
        every = _bi.max(1, int((total or 100) / 20))
        for i, item in enumerate(iterable, 1):
            if i == 1 or i % every == 0 or (total is not None and i == total):
                print("%s %s%s" % (desc, i, "/%s" % total if total else ""))
            yield item
    return _gen()


def display_df(df, n=30):
    try:
        display(df.head(n))
    except Exception:
        print(df.head(n).to_string(index=False))


def save_show(fig, filename):
    fig.tight_layout()
    fig.savefig(FIG_DIR / filename, dpi=140, bbox_inches="tight")
    plt.show()
    plt.close(fig)


## 1. 实验配置


In [ ]:
PROJECT_DIR = Path.cwd()
OUT_DIR = PROJECT_DIR / "csi800_ml_v85_market_state_outputs"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

DATA_CANDIDATES = [
    Path("train_csi800_factor_v40_data_enhancement_20190101_20260531.csv"),
    Path("train_csi800_factor_v40_data_enhancement.csv"),
    Path("data/train_csi800_factor_v40_data_enhancement_20190101_20260531.csv"),
    Path("data/train_csi800_factor_v40_data_enhancement.csv"),
    PROJECT_DIR / "train_csi800_factor_v40_data_enhancement_20190101_20260531.csv",
    PROJECT_DIR / "train_csi800_factor_v40_data_enhancement.csv",
]
DATA_PATH_OVERRIDE = None

TARGET_COL = "alpha_1m"
STOCK_COL = "stock"
DATE_COL = "rebalance_date"
INDUSTRY_COL = "industry_bucket"
LABEL_BOUNDARY_MODE = "legacy_rebalance"

FIXED_ITER = 120
SEED = 42
CORR_THRESHOLD = 0.70
REBUILD_MARKET_STATE = False
STATE_CACHE_CANDIDATES = [
    Path("csi800_ml_v85_market_state_outputs/v85_market_state_cache.csv"),
    Path("v85_market_state_cache.csv"),
    PROJECT_DIR / "v85_market_state_cache.csv",
]
INDEX_SPECS = [
    ("csi300", "000300.XSHG"),
    ("csi500", "000905.XSHG"),
    ("csi800", "000906.XSHG"),
]
STATE_PRICE_BATCH_SIZE = 300
STATE_API_RETRIES = 3
MAX_STATE_CELL_MISSING_RATIO = 0.10
TOP_N_CANDIDATES = 30
STOCK_NUM = 8
BOARD_CAPS = {"chinext": 3, "star": 2}
BUCKET_N = 10
BOOTSTRAP_N = 1000
BOOTSTRAP_BLOCK_MONTHS = 6

RUN_VARIANTS = [
    "A0_full_v46",
    "M1_true_state",
    "P1_placebo_state",
]

FOLD_PLAN = [
    {"fold_id": "cutoff202112", "train_start": "2019-01-01", "train_end": "2021-12-31", "test_start": "2022-01-01", "test_end": "2022-12-31"},
    {"fold_id": "cutoff202212", "train_start": "2019-01-01", "train_end": "2022-12-31", "test_start": "2023-01-01", "test_end": "2023-12-31"},
    {"fold_id": "cutoff202312", "train_start": "2019-01-01", "train_end": "2023-12-31", "test_start": "2024-01-01", "test_end": "2024-12-31"},
    {"fold_id": "cutoff202412", "train_start": "2019-01-01", "train_end": "2024-12-31", "test_start": "2025-01-01", "test_end": "2025-12-31"},
    {"fold_id": "cutoff202512", "train_start": "2019-01-01", "train_end": "2025-12-31", "test_start": "2026-01-01", "test_end": "2026-12-31"},
]

SMOKE_TEST = False
if SMOKE_TEST:
    RUN_VARIANTS = RUN_VARIANTS[:2]
    FOLD_PLAN = FOLD_PLAN[:1]
    BOOTSTRAP_N = 100

COLORS = {
    "A0_full_v46": "#2f5597",
    "M1_true_state": "#00a087",
    "P1_placebo_state": "#e64b35",
}

print("OUT_DIR:", OUT_DIR)
print("RUN_VARIANTS:", RUN_VARIANTS)
print("FOLDS:", [x["fold_id"] for x in FOLD_PLAN])


## 2. V46 full 特征和固定模型参数


In [ ]:
def unique_keep_order(cols):
    seen = set()
    out = []
    for col in cols:
        if col not in seen:
            out.append(col)
            seen.add(col)
    return out


BASE_FACTOR_COLS = [
    "cash_flow_to_price_ratio", "book_to_price_ratio", "earnings_yield", "sales_to_price_ratio",
    "cash_earnings_to_price_ratio", "earnings_to_price_ratio", "roe_ttm", "roa_ttm",
    "gross_profit_ttm", "operating_profit_to_total_profit", "net_operate_cash_flow_to_total_liability",
    "net_operating_cash_flow_coverage", "adjusted_profit_to_total_profit", "ACCA", "growth",
    "net_working_capital", "operating_profit_per_share", "net_operate_cash_flow_per_share",
    "total_operating_revenue_per_share", "super_quick_ratio", "MLEV", "debt_to_equity_ratio",
    "debt_to_tangible_equity_ratio", "momentum", "Rank1M", "sharpe_ratio_60", "Variance20",
    "liquidity", "beta", "ATR6", "MFI14", "DAVOL10", "VOL10", "VMACD", "VOSC",
    "Skewness20", "Kurtosis20",
]
HYBRID_LIGHT_EXTRA_COLS = [
    "liq_money_ratio_20_60", "liq_paused_count_20", "px_close_to_ma60", "px_drawdown_60",
    "ts_cash_flow_to_price_ratio_rank_mean_3m", "ts_Rank1M_rank_chg_1m",
]
FULL_FEATURE_COLS = unique_keep_order(BASE_FACTOR_COLS + HYBRID_LIGHT_EXTRA_COLS)

BASE_PARAMS_FF10 = {
    "objective": "regression",
    "metric": "l2",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 200,
    "feature_fraction": 1.0,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 0.3,
    "verbose": -1,
}

STATE_FEATURE_COLS = []
for prefix, _ in INDEX_SPECS:
    STATE_FEATURE_COLS.extend([
        "state_%s_ret20" % prefix,
        "state_%s_ret60" % prefix,
        "state_%s_vol20" % prefix,
        "state_%s_money20_to60" % prefix,
    ])
STATE_FEATURE_COLS.extend([
    "state_csi800_above_ma20",
    "state_csi800_above_ma60",
    "state_csi800_ret20_positive_ratio",
    "state_csi800_ret20_dispersion",
])

VARIANT_MANIFEST = [
    {"variant": "A0_full_v46", "state_mode": "none"},
    {"variant": "M1_true_state", "state_mode": "true"},
    {"variant": "P1_placebo_state", "state_mode": "placebo"},
]
VARIANT_MANIFEST = [x for x in VARIANT_MANIFEST if x["variant"] in set(RUN_VARIANTS)]
variant_manifest_df = pd.DataFrame(VARIANT_MANIFEST)
variant_manifest_df.to_csv(OUT_DIR / "v85_variant_manifest.csv", index=False)
display_df(variant_manifest_df, 20)


## 3. 数据、标签和训练工具


In [ ]:
def resolve_data_path():
    if DATA_PATH_OVERRIDE:
        p = Path(DATA_PATH_OVERRIDE)
        if p.exists():
            return p
        raise IOError("DATA_PATH_OVERRIDE not found: %s" % p)
    for raw in DATA_CANDIDATES:
        p = Path(raw)
        if p.exists():
            return p
    searched = [str(Path(x).resolve()) for x in DATA_CANDIDATES]
    raise IOError("training data csv not found: %s" % searched)


def safe_to_datetime(df, cols):
    out = df.copy()
    for col in cols:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col], errors="coerce").dt.normalize()
    return out


def safe_rank_ic(a, b):
    s = pd.DataFrame({"a": np.asarray(a, dtype=float), "b": np.asarray(b, dtype=float)})
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 3 or s["a"].nunique() < 2 or s["b"].nunique() < 2:
        return np.nan
    return float(s["a"].rank(method="average").corr(s["b"].rank(method="average")))


def safe_pearson_ic(a, b):
    s = pd.DataFrame({"a": np.asarray(a, dtype=float), "b": np.asarray(b, dtype=float)})
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 3 or s["a"].nunique() < 2 or s["b"].nunique() < 2:
        return np.nan
    return float(s["a"].corr(s["b"]))


def load_dataset(path):
    df = pd.read_csv(path)
    df = safe_to_datetime(df, [DATE_COL, "feature_date", "next_date"])
    if STOCK_COL not in df.columns:
        for alt in ["code", "security", "order_book_id"]:
            if alt in df.columns:
                df = df.rename(columns={alt: STOCK_COL})
                break
    if TARGET_COL not in df.columns:
        if "raw_return_1m" in df.columns and "benchmark_csi800_1m" in df.columns:
            df[TARGET_COL] = pd.to_numeric(df["raw_return_1m"], errors="coerce") - pd.to_numeric(df["benchmark_csi800_1m"], errors="coerce")
        else:
            raise ValueError("target column not found: " + TARGET_COL)
    if INDUSTRY_COL not in df.columns:
        df[INDUSTRY_COL] = "UNKNOWN"
    if "feature_date" not in df.columns:
        df["feature_date"] = df[DATE_COL]
    if "next_date" not in df.columns:
        df["next_date"] = df[DATE_COL]
    need = [STOCK_COL, DATE_COL, TARGET_COL, INDUSTRY_COL, "feature_date", "next_date"]
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise ValueError("dataset missing columns: " + ",".join(missing))
    missing_features = [c for c in FULL_FEATURE_COLS if c not in df.columns]
    if missing_features:
        raise ValueError("missing full_v46 features: " + ",".join(missing_features))
    df[STOCK_COL] = df[STOCK_COL].astype(str)
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    df = df.dropna(subset=[STOCK_COL, DATE_COL, TARGET_COL]).copy()
    df = df.sort_values([DATE_COL, STOCK_COL]).reset_index(drop=True)
    for col in progress_iter(FULL_FEATURE_COLS + [TARGET_COL], total=len(FULL_FEATURE_COLS) + 1, desc="compact float32"):
        df[col] = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan).astype(np.float32)
    return df


def make_train_df(df, fold):
    start = pd.Timestamp(fold["train_start"])
    end = pd.Timestamp(fold["train_end"])
    mask = (df[DATE_COL] >= start) & (df[DATE_COL] <= end)
    if LABEL_BOUNDARY_MODE == "label_end_safe":
        mask = mask & (df["next_date"] <= end)
    elif LABEL_BOUNDARY_MODE != "legacy_rebalance":
        raise ValueError("unknown LABEL_BOUNDARY_MODE: " + str(LABEL_BOUNDARY_MODE))
    return df.loc[mask].copy()


def make_test_df(df, fold):
    start = pd.Timestamp(fold["test_start"])
    end = pd.Timestamp(fold["test_end"])
    return df.loc[(df[DATE_COL] >= start) & (df[DATE_COL] <= end)].copy()


def build_corr_components(train_df, feature_cols, threshold):
    from collections import defaultdict
    corr = train_df[feature_cols].corr()
    graph = defaultdict(list)
    for i in range(len(feature_cols)):
        for j in range(i + 1, len(feature_cols)):
            v = corr.iloc[i, j]
            if not pd.isnull(v) and abs(v) > threshold:
                graph[feature_cols[i]].append(feature_cols[j])
                graph[feature_cols[j]].append(feature_cols[i])
    for col in feature_cols:
        graph[col]
    visited = set()
    comps = []
    def dfs(x, comp):
        visited.add(x)
        comp.append(x)
        for y in graph[x]:
            if y not in visited:
                dfs(y, comp)
    for col in feature_cols:
        if col not in visited:
            comp = []
            dfs(col, comp)
            comps.append(comp)
    return comps


def select_features_train_only(train_df, candidate_cols):
    cols = unique_keep_order([c for c in candidate_cols if c in train_df.columns])
    missing = train_df[cols].isnull().sum().to_dict()
    keep = []
    remove = []
    for comp in build_corr_components(train_df, cols, CORR_THRESHOLD):
        if len(comp) == 1:
            keep.append(comp[0])
        else:
            ordered = _bi.sorted(comp, key=lambda x: (missing[x], x))
            keep.append(ordered[0])
            remove.extend(ordered[1:])
    if len(keep) == 0:
        raise ValueError("no usable full_v46 feature")
    return keep, remove


def prepare_x(df, feature_cols, fill_values=None):
    X = df.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan).copy()
    if fill_values is None:
        fill_values = X.median().replace([np.inf, -np.inf], np.nan).fillna(0)
    X = X.fillna(fill_values).fillna(0)
    return X, fill_values


def chunked(values, size):
    values = list(values)
    for i in range(0, len(values), int(size)):
        yield values[i:i + int(size)]


def api_retry(fn, desc):
    last_error = None
    for attempt in range(int(STATE_API_RETRIES)):
        try:
            return fn()
        except Exception as exc:
            last_error = exc
            print("API retry %s/%s for %s: %s" % (attempt + 1, STATE_API_RETRIES, desc, exc))
            time.sleep(1 + attempt)
    raise RuntimeError("API failed after retries for %s: %s" % (desc, last_error))


def normalize_price_long(raw, default_code=None):
    if raw is None:
        return pd.DataFrame(columns=["time", "code", "close", "money"])
    out = raw.copy() if isinstance(raw, pd.DataFrame) else pd.DataFrame(raw)
    if isinstance(out.index, pd.MultiIndex):
        for level in range(out.index.nlevels):
            out["__idx_%s" % level] = out.index.get_level_values(level)
        out = out.reset_index(drop=True)
    elif not isinstance(out.index, pd.RangeIndex):
        out["__index__"] = out.index
        out = out.reset_index(drop=True)

    time_col = None
    for col in ["time", "date", "datetime", "__index__", "__idx_0", "__idx_1"]:
        if col in out.columns:
            parsed = pd.to_datetime(out[col], errors="coerce")
            if parsed.notnull().mean() >= 0.80:
                time_col = col
                out["time"] = parsed.dt.normalize()
                break
    if time_col is None:
        raise ValueError("cannot identify time column in get_price result: %s" % list(out.columns))

    code_col = None
    for col in ["code", "security", "order_book_id", "__idx_1", "__idx_0"]:
        if col in out.columns and col != time_col:
            values = out[col].astype(str)
            if values.str.contains("XSHG|XSHE|\\.").mean() >= 0.50:
                code_col = col
                break
    if code_col is not None:
        out["code"] = out[code_col].astype(str)
    elif default_code is not None:
        out["code"] = str(default_code)
    else:
        raise ValueError("cannot identify security column in get_price result: %s" % list(out.columns))

    for col in ["close", "money"]:
        if col not in out.columns:
            out[col] = np.nan
        out[col] = pd.to_numeric(out[col], errors="coerce")
    out = out[["time", "code", "close", "money"]].dropna(subset=["time", "code"])
    return out.sort_values(["time", "code"]).drop_duplicates(["time", "code"], keep="last")


def fetch_index_histories(feature_dates):
    if get_price is None:
        raise RuntimeError("get_price is unavailable; run this notebook in JoinQuant or provide v85_market_state_cache.csv")
    start = pd.Timestamp(_bi.min(feature_dates)) - pd.Timedelta(days=140)
    end = pd.Timestamp(_bi.max(feature_dates))
    histories = {}
    status_rows = []
    for prefix, code_value in progress_iter(INDEX_SPECS, total=len(INDEX_SPECS), desc="fetch index state history"):
        raw = api_retry(
            lambda code_value=code_value: get_price(
                code_value, start_date=start, end_date=end, frequency="daily",
                fields=["close", "money"], panel=False,
            ),
            "index history " + code_value,
        )
        long_df = normalize_price_long(raw, code_value)
        histories[prefix] = long_df.set_index("time")[["close", "money"]].sort_index()
        status_rows.append({
            "scope": "index", "name": prefix, "code": code_value, "feature_date": pd.NaT,
            "rows": int(len(long_df)), "status": "ok" if len(long_df) >= 61 else "short_history",
        })
    return histories, status_rows


def calculate_index_state(history, feature_date, prefix):
    window = history.loc[history.index <= pd.Timestamp(feature_date)].tail(61)
    row = {}
    if len(window) < 61:
        for suffix in ["ret20", "ret60", "vol20", "money20_to60"]:
            row["state_%s_%s" % (prefix, suffix)] = np.nan
        return row, "short_history"
    close = pd.to_numeric(window["close"], errors="coerce")
    money = pd.to_numeric(window["money"], errors="coerce")
    daily_ret = close.pct_change().replace([np.inf, -np.inf], np.nan)
    row["state_%s_ret20" % prefix] = float(close.iloc[-1] / close.iloc[-21] - 1.0) if close.iloc[-21] > 0 else np.nan
    row["state_%s_ret60" % prefix] = float(close.iloc[-1] / close.iloc[-61] - 1.0) if close.iloc[-61] > 0 else np.nan
    row["state_%s_vol20" % prefix] = float(daily_ret.tail(20).std())
    money60 = float(money.tail(60).mean())
    row["state_%s_money20_to60" % prefix] = float(money.tail(20).mean() / money60 - 1.0) if money60 > 0 else np.nan
    return row, "ok"


def fetch_csi800_breadth(feature_date):
    if get_index_stocks is None:
        raise RuntimeError("get_index_stocks is unavailable")
    members = api_retry(
        lambda: get_index_stocks("000906.XSHG", date=pd.Timestamp(feature_date)),
        "CSI800 members %s" % pd.Timestamp(feature_date).date(),
    )
    members = unique_keep_order([str(x) for x in members])
    pieces = []
    batches = list(chunked(members, STATE_PRICE_BATCH_SIZE))
    for batch in progress_iter(batches, total=len(batches), desc="breadth %s" % pd.Timestamp(feature_date).strftime("%Y-%m-%d"), leave=False):
        raw = api_retry(
            lambda batch=batch: get_price(
                batch, end_date=pd.Timestamp(feature_date), count=61, frequency="daily",
                fields=["close"], panel=False,
            ),
            "breadth prices %s" % pd.Timestamp(feature_date).date(),
        )
        pieces.append(normalize_price_long(raw))
        del raw
    long_df = pd.concat(pieces, ignore_index=True) if pieces else pd.DataFrame()
    if len(long_df) == 0:
        return dict((c, np.nan) for c in STATE_FEATURE_COLS[-4:]), 0, "empty"
    pivot = long_df.pivot_table(index="time", columns="code", values="close", aggfunc="last").sort_index().tail(61)
    if len(pivot) < 61:
        return dict((c, np.nan) for c in STATE_FEATURE_COLS[-4:]), int(pivot.shape[1]), "short_history"
    latest = pivot.iloc[-1]
    ma20 = pivot.tail(20).mean()
    ma60 = pivot.tail(60).mean()
    base20 = pivot.iloc[-21]
    valid20 = latest.notnull() & ma20.notnull()
    valid60 = latest.notnull() & ma60.notnull()
    valid_ret = latest.notnull() & base20.notnull() & (base20 > 0)
    ret20 = latest[valid_ret] / base20[valid_ret] - 1.0
    row = {
        "state_csi800_above_ma20": float((latest[valid20] > ma20[valid20]).mean()) if valid20.sum() else np.nan,
        "state_csi800_above_ma60": float((latest[valid60] > ma60[valid60]).mean()) if valid60.sum() else np.nan,
        "state_csi800_ret20_positive_ratio": float((ret20 > 0).mean()) if len(ret20) else np.nan,
        "state_csi800_ret20_dispersion": float(ret20.std()) if len(ret20) >= 3 else np.nan,
    }
    status = "ok" if int(valid60.sum()) >= 400 else "low_coverage"
    return row, int(valid60.sum()), status


def build_market_state_cache(feature_dates):
    feature_dates = _bi.sorted(set(pd.Timestamp(x).normalize() for x in feature_dates if not pd.isnull(x)))
    histories, status_rows = fetch_index_histories(feature_dates)
    rows = []
    for feature_date in progress_iter(feature_dates, total=len(feature_dates), desc="build market state cache"):
        row = {"feature_date": pd.Timestamp(feature_date)}
        for prefix, code_value in INDEX_SPECS:
            part, status = calculate_index_state(histories[prefix], feature_date, prefix)
            row.update(part)
            status_rows.append({
                "scope": "index_state", "name": prefix, "code": code_value,
                "feature_date": pd.Timestamp(feature_date), "rows": 61, "status": status,
            })
        breadth, coverage, status = fetch_csi800_breadth(feature_date)
        row.update(breadth)
        rows.append(row)
        status_rows.append({
            "scope": "breadth", "name": "csi800", "code": "000906.XSHG",
            "feature_date": pd.Timestamp(feature_date), "rows": int(coverage), "status": status,
        })
        gc.collect()
    return pd.DataFrame(rows), pd.DataFrame(status_rows)


def select_state_features_train_only(train_df):
    monthly = train_df[[DATE_COL] + STATE_FEATURE_COLS].drop_duplicates(DATE_COL).sort_values(DATE_COL)
    return select_features_train_only(monthly, STATE_FEATURE_COLS)


def apply_placebo_state(train_df, test_df, state_cols, seed):
    train_out = train_df.copy()
    test_out = test_df.copy()
    monthly = train_out[[DATE_COL] + state_cols].drop_duplicates(DATE_COL).sort_values(DATE_COL)
    if len(monthly) < 2:
        raise ValueError("placebo requires at least two training months")
    train_dates = monthly[DATE_COL].tolist()
    source_values = monthly[state_cols].values.copy()
    rng = np.random.RandomState(int(seed))
    perm = rng.permutation(len(monthly))
    train_sources = [train_dates[int(i)] for i in perm]
    test_dates = _bi.sorted(test_out[DATE_COL].dropna().unique().tolist())
    sampled = rng.randint(0, len(monthly), size=len(test_dates))
    test_sources = [train_dates[int(i)] for i in sampled]
    audit_rows = []
    for j, col in enumerate(state_cols):
        train_map = pd.Series(source_values[perm, j], index=train_dates)
        test_map = pd.Series(source_values[sampled, j], index=test_dates)
        train_out[col] = train_out[DATE_COL].map(train_map)
        test_out[col] = test_out[DATE_COL].map(test_map)
    for dt, source_dt in zip(train_dates, train_sources):
        audit_rows.append({"sample": "train", DATE_COL: pd.Timestamp(dt), "source_state_date": pd.Timestamp(source_dt)})
    for dt, source_dt in zip(test_dates, test_sources):
        audit_rows.append({"sample": "test", DATE_COL: pd.Timestamp(dt), "source_state_date": pd.Timestamp(source_dt)})
    return train_out, test_out, pd.DataFrame(audit_rows)


def train_model(train_df, feature_cols):
    work = train_df.sort_values([DATE_COL, STOCK_COL]).dropna(subset=[TARGET_COL]).copy()
    X, fill_values = prepare_x(work, feature_cols, None)
    y = pd.to_numeric(work[TARGET_COL], errors="coerce")
    params = dict(BASE_PARAMS_FF10)
    params["seed"] = SEED
    dataset = lgb.Dataset(X[feature_cols], label=np.asarray(y))
    model = lgb.train(params, dataset, num_boost_round=_bi.max(1, int(FIXED_ITER)))
    train_pred = np.asarray(model.predict(X[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)
    train_rank_ic = safe_rank_ic(train_pred, y)
    return {
        "model": model, "fill_values": fill_values, "train_rows": len(work),
        "train_rank_ic": train_rank_ic,
        "target_mean": float(y.mean()), "target_std": float(y.std()),
    }


def score_model(df, trained, feature_cols):
    X, _ = prepare_x(df, feature_cols, trained["fill_values"])
    return np.asarray(trained["model"].predict(X[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)


## 4. 排序、分类、组合和统计指标


In [ ]:
def average_precision_at_k(pred_order, relevant_set, k):
    if len(relevant_set) == 0:
        return np.nan
    hits = 0
    score = 0.0
    top = pred_order[:_bi.min(k, len(pred_order))]
    for i, stock in enumerate(top, 1):
        if stock in relevant_set:
            hits += 1
            score += hits / float(i)
    denom = float(_bi.min(len(relevant_set), k))
    return score / denom if denom > 0 else np.nan


def dcg_at_k(gains, k):
    vals = np.asarray(gains[:_bi.min(k, len(gains))], dtype=float)
    if len(vals) == 0:
        return np.nan
    denom = np.log2(np.arange(2, len(vals) + 2))
    return float(np.nansum(vals / denom))


def ndcg_at_k(pred_order, gain_map, k):
    valid_map = {}
    for stock, gain in gain_map.items():
        if not pd.isnull(gain) and np.isfinite(float(gain)):
            valid_map[stock] = float(gain)
    if len(valid_map) == 0:
        return np.nan
    # Use one common non-negative relevance scale for predicted and ideal lists.
    # Shifting each Top-K list separately can produce an invalid NDCG greater than 1.
    global_floor = _bi.min(valid_map.values())
    relevance_map = dict((stock, gain - global_floor) for stock, gain in valid_map.items())
    pred_gains = [relevance_map.get(s, np.nan) for s in pred_order]
    ideal_gains = _bi.sorted(relevance_map.values(), reverse=True)
    dcg = dcg_at_k(pred_gains, k)
    idcg = dcg_at_k(ideal_gains, k)
    if pd.isnull(dcg) or pd.isnull(idcg) or idcg <= 0:
        return np.nan
    return dcg / idcg


def roc_auc_binary(y_true, scores):
    d = pd.DataFrame({"y": np.asarray(y_true, dtype=float), "s": np.asarray(scores, dtype=float)})
    d = d.replace([np.inf, -np.inf], np.nan).dropna()
    n_pos = int((d["y"] > 0).sum())
    n_neg = int((d["y"] <= 0).sum())
    if n_pos == 0 or n_neg == 0:
        return np.nan
    ranks = d["s"].rank(method="average", ascending=True)
    rank_sum = float(ranks[d["y"] > 0].sum())
    return (rank_sum - n_pos * (n_pos + 1) / 2.0) / float(n_pos * n_neg)


def average_precision_binary(y_true, scores):
    d = pd.DataFrame({"y": np.asarray(y_true, dtype=float), "s": np.asarray(scores, dtype=float)})
    d = d.replace([np.inf, -np.inf], np.nan).dropna().sort_values("s", ascending=False)
    y = (d["y"].values > 0).astype(int)
    n_pos = int(y.sum())
    if n_pos == 0:
        return np.nan
    tp = np.cumsum(y)
    precision = tp / np.arange(1, len(y) + 1, dtype=float)
    return float(precision[y == 1].sum() / n_pos)


def binary_curve_points(y_true, scores):
    d = pd.DataFrame({"y": np.asarray(y_true, dtype=float), "s": np.asarray(scores, dtype=float)})
    d = d.replace([np.inf, -np.inf], np.nan).dropna().sort_values("s", ascending=False)
    y = (d["y"].values > 0).astype(int)
    n_pos = int(y.sum())
    n_neg = int(len(y) - n_pos)
    if n_pos == 0 or n_neg == 0:
        return pd.DataFrame()
    tp = np.cumsum(y).astype(float)
    fp = np.cumsum(1 - y).astype(float)
    recall = tp / float(n_pos)
    precision = tp / np.arange(1, len(y) + 1, dtype=float)
    tpr = recall
    fpr = fp / float(n_neg)
    return pd.DataFrame({"fpr": np.r_[0.0, fpr, 1.0], "tpr": np.r_[0.0, tpr, 1.0], "recall": np.r_[0.0, recall, 1.0], "precision": np.r_[1.0, precision, float(n_pos) / len(y)]})


def board_name(stock):
    code6 = str(stock).split(".")[0]
    if code6.startswith(("300", "301")):
        return "chinext"
    if code6.startswith("688"):
        return "star"
    return "main"


def select_top8_board_cap(month_df):
    ordered = month_df.sort_values("score", ascending=False)[STOCK_COL].astype(str).tolist()
    selected = []
    counts = {"chinext": 0, "star": 0, "main": 0}
    for stock in ordered[:_bi.min(TOP_N_CANDIDATES, len(ordered))]:
        board = board_name(stock)
        cap = BOARD_CAPS.get(board, STOCK_NUM)
        if counts.get(board, 0) >= cap:
            continue
        selected.append(stock)
        counts[board] = counts.get(board, 0) + 1
        if len(selected) >= STOCK_NUM:
            break
    return selected


def calc_nav(ret_series):
    s = pd.Series(ret_series).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return (1.0 + s).cumprod()


def calc_mdd(ret_series):
    nav = calc_nav(ret_series)
    if len(nav) == 0:
        return np.nan
    return float((nav / nav.cummax() - 1.0).min())


def annualized_return(ret_series):
    s = pd.Series(ret_series).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) == 0:
        return np.nan
    total = float((1.0 + s).prod())
    if total <= 0:
        return np.nan
    return total ** (12.0 / len(s)) - 1.0


def icir(series):
    s = pd.to_numeric(pd.Series(series), errors="coerce").dropna()
    if len(s) < 2 or float(s.std()) <= 1e-12:
        return np.nan
    return float(s.mean() / s.std() * math.sqrt(12.0))


def moving_block_bootstrap_mean(values, n_sim=BOOTSTRAP_N, block=BOOTSTRAP_BLOCK_MONTHS, seed=SEED):
    x = np.asarray(pd.Series(values).dropna(), dtype=float)
    n = len(x)
    if n < 3:
        return np.nan, np.nan, np.nan
    rng = np.random.RandomState(seed)
    block = int(_bi.max(1, _bi.min(block, n)))
    sims = []
    starts = np.arange(n)
    for _ in range(int(n_sim)):
        sampled = []
        while len(sampled) < n:
            st = int(rng.choice(starts))
            sampled.extend([x[(st + j) % n] for j in range(block)])
        sims.append(float(np.mean(sampled[:n])))
    arr = np.asarray(sims, dtype=float)
    return float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5)), float((arr > 0).mean())


## 5. 加载数据并构建点时市场状态缓存


In [ ]:
DATA_PATH = resolve_data_path()
df_all = load_dataset(DATA_PATH)

date_map = df_all[[DATE_COL, "feature_date"]].drop_duplicates().copy()
date_conflicts = date_map.groupby(DATE_COL)["feature_date"].nunique()
if int((date_conflicts > 1).sum()) > 0:
    raise ValueError("one rebalance_date maps to multiple feature_date values")
feature_dates = _bi.sorted(date_map["feature_date"].dropna().unique().tolist())

cache_path = None
if not REBUILD_MARKET_STATE:
    for raw in STATE_CACHE_CANDIDATES:
        candidate = Path(raw)
        if candidate.exists():
            cache_path = candidate
            break
if cache_path is not None:
    print("load market state cache:", cache_path)
    market_state_df = safe_to_datetime(pd.read_csv(cache_path), ["feature_date"])
    state_fetch_status_df = pd.DataFrame([{
        "scope": "cache", "name": str(cache_path), "code": "", "feature_date": pd.NaT,
        "rows": int(len(market_state_df)), "status": "loaded",
    }])
else:
    print("build market state from JoinQuant API")
    market_state_df, state_fetch_status_df = build_market_state_cache(feature_dates)

missing_state_cols = [c for c in STATE_FEATURE_COLS if c not in market_state_df.columns]
if missing_state_cols:
    raise ValueError("market state cache missing columns: " + ",".join(missing_state_cols))
market_state_df = safe_to_datetime(market_state_df, ["feature_date"])
available_dates = set(market_state_df["feature_date"].dropna().tolist())
missing_feature_dates = [pd.Timestamp(x) for x in feature_dates if pd.Timestamp(x) not in available_dates]
if cache_path is not None and len(missing_feature_dates):
    if get_price is None or get_index_stocks is None:
        raise RuntimeError("market state cache is stale by %s dates and JoinQuant API is unavailable" % len(missing_feature_dates))
    print("incrementally append %s new market-state dates" % len(missing_feature_dates))
    incremental_df, incremental_status_df = build_market_state_cache(missing_feature_dates)
    market_state_df = pd.concat([market_state_df, incremental_df], ignore_index=True)
    state_fetch_status_df = pd.concat([state_fetch_status_df, incremental_status_df], ignore_index=True)
market_state_df = market_state_df.drop_duplicates("feature_date", keep="last").sort_values("feature_date")
for col in progress_iter(STATE_FEATURE_COLS, total=len(STATE_FEATURE_COLS), desc="compact state float32"):
    market_state_df[col] = pd.to_numeric(market_state_df[col], errors="coerce").replace([np.inf, -np.inf], np.nan).astype(np.float32)
market_state_df.to_csv(OUT_DIR / "v85_market_state_cache.csv", index=False)
state_fetch_status_df.to_csv(OUT_DIR / "v85_market_state_fetch_status.csv", index=False)

requested = pd.DataFrame({"feature_date": feature_dates})
requested["feature_date"] = pd.to_datetime(requested["feature_date"]).dt.normalize()
state_availability_df = pd.merge(requested, market_state_df[["feature_date"] + STATE_FEATURE_COLS], on="feature_date", how="left")
state_availability_rows = []
for col in STATE_FEATURE_COLS:
    state_availability_rows.append({
        "feature": col, "requested_dates": int(len(state_availability_df)),
        "missing_dates": int(state_availability_df[col].isnull().sum()),
        "missing_ratio": float(state_availability_df[col].isnull().mean()),
        "std": float(pd.to_numeric(state_availability_df[col], errors="coerce").std()),
    })
state_availability_audit_df = pd.DataFrame(state_availability_rows)
overall_missing_ratio = float(state_availability_df[STATE_FEATURE_COLS].isnull().sum().sum()) / float(_bi.max(1, len(state_availability_df) * len(STATE_FEATURE_COLS)))
if overall_missing_ratio > MAX_STATE_CELL_MISSING_RATIO:
    raise ValueError("market state missing ratio %.4f exceeds %.4f" % (overall_missing_ratio, MAX_STATE_CELL_MISSING_RATIO))

df_all = pd.merge(df_all, market_state_df[["feature_date"] + STATE_FEATURE_COLS], on="feature_date", how="left")
for col in STATE_FEATURE_COLS:
    df_all[col] = pd.to_numeric(df_all[col], errors="coerce").astype(np.float32)

audit_rows = [
    {"check": "rows", "value": int(len(df_all))},
    {"check": "months", "value": int(df_all[DATE_COL].nunique())},
    {"check": "date_min", "value": str(df_all[DATE_COL].min())},
    {"check": "date_max", "value": str(df_all[DATE_COL].max())},
    {"check": "duplicate_stock_date", "value": int(df_all.duplicated([STOCK_COL, DATE_COL]).sum())},
    {"check": "v46_feature_count", "value": int(len(FULL_FEATURE_COLS))},
    {"check": "state_feature_count", "value": int(len(STATE_FEATURE_COLS))},
    {"check": "state_dates", "value": int(len(market_state_df))},
    {"check": "state_cell_missing_ratio", "value": overall_missing_ratio},
    {"check": "state_cache_source", "value": str(cache_path) if cache_path is not None else "JoinQuant API"},
    {"check": "lightgbm_version", "value": str(getattr(lgb, "__version__", "unknown"))},
]
data_audit_df = pd.DataFrame(audit_rows)
data_audit_df.to_csv(OUT_DIR / "v85_data_audit.csv", index=False)
state_availability_audit_df.to_csv(OUT_DIR / "v85_state_availability_audit.csv", index=False)
print("DATA_PATH:", DATA_PATH)
print("loaded:", df_all.shape)
display_df(data_audit_df, 20)
display_df(state_availability_audit_df, 30)
display_df(market_state_df, 10)


## 6. 逐 fold 训练 A0、真实状态与 placebo 状态


In [ ]:
score_parts = []
model_meta_rows = []
feature_importance_rows = []
placebo_audit_parts = []
total_models = len(FOLD_PLAN) * len(VARIANT_MANIFEST)
model_counter = 0

for fold_index, fold in enumerate(progress_iter(FOLD_PLAN, total=len(FOLD_PLAN), desc="walk-forward folds")):
    train_df = make_train_df(df_all, fold)
    test_df = make_test_df(df_all, fold)
    if train_df.empty or test_df.empty:
        print("skip empty fold", fold["fold_id"], train_df.shape, test_df.shape)
        del train_df, test_df
        gc.collect()
        continue
    base_feature_cols, removed_base_cols = select_features_train_only(train_df, FULL_FEATURE_COLS)
    state_feature_cols, removed_state_cols = select_state_features_train_only(train_df)
    for variant in VARIANT_MANIFEST:
        model_counter += 1
        print("train model %s/%s: %s %s" % (model_counter, total_models, fold["fold_id"], variant["variant"]))
        if variant["state_mode"] == "none":
            variant_train_df = train_df
            variant_test_df = test_df
            feature_cols = list(base_feature_cols)
        elif variant["state_mode"] == "true":
            variant_train_df = train_df
            variant_test_df = test_df
            feature_cols = list(base_feature_cols) + list(state_feature_cols)
        elif variant["state_mode"] == "placebo":
            variant_train_df, variant_test_df, placebo_part = apply_placebo_state(
                train_df, test_df, STATE_FEATURE_COLS, SEED + fold_index * 1009,
            )
            placebo_part["fold_id"] = fold["fold_id"]
            placebo_part["variant"] = variant["variant"]
            placebo_audit_parts.append(placebo_part)
            feature_cols = list(base_feature_cols) + list(state_feature_cols)
        else:
            raise ValueError("unknown state_mode: " + str(variant["state_mode"]))

        trained = train_model(variant_train_df, feature_cols)
        pred = score_model(variant_test_df, trained, feature_cols)
        gain_values = trained["model"].feature_importance(importance_type="gain")
        split_values = trained["model"].feature_importance(importance_type="split")
        for i_feature, feature_name in enumerate(feature_cols):
            feature_importance_rows.append({
                "fold_id": fold["fold_id"], "variant": variant["variant"],
                "feature": feature_name, "importance_gain": float(gain_values[i_feature]),
                "importance_split": float(split_values[i_feature]),
                "feature_group": "market_state" if feature_name in set(STATE_FEATURE_COLS) else "v46_stock_factor",
            })
        keep_cols = [STOCK_COL, DATE_COL, TARGET_COL, INDUSTRY_COL, "next_date"]
        keep_cols = [c for c in keep_cols if c in variant_test_df.columns]
        part = variant_test_df[keep_cols].copy()
        part["score"] = pred.astype(np.float32)
        part["variant"] = variant["variant"]
        part["fold_id"] = fold["fold_id"]
        part["train_end"] = pd.Timestamp(fold["train_end"])
        score_parts.append(part)
        model_meta_rows.append({
            "fold_id": fold["fold_id"], "variant": variant["variant"],
            "train_start": fold["train_start"], "train_end": fold["train_end"],
            "test_start": fold["test_start"], "test_end": fold["test_end"],
            "train_months": int(train_df[DATE_COL].nunique()), "train_rows": int(len(train_df)),
            "test_months": int(test_df[DATE_COL].nunique()), "test_rows": int(len(test_df)),
            "base_feature_count": int(len(base_feature_cols)), "state_feature_count": int(len(feature_cols) - len(base_feature_cols)),
            "feature_count": int(len(feature_cols)), "removed_base_feature_count": int(len(removed_base_cols)),
            "removed_state_feature_count": int(len(removed_state_cols)),
            "train_rank_ic": trained["train_rank_ic"],
            "trained_rows": int(trained["train_rows"]),
            "trained_target_mean": trained["target_mean"], "trained_target_std": trained["target_std"],
            "state_mode": variant["state_mode"],
            "feature_cols": ",".join(feature_cols),
            "state_feature_cols": ",".join([c for c in feature_cols if c in set(STATE_FEATURE_COLS)]),
            "removed_base_features": ",".join(removed_base_cols),
            "removed_state_features": ",".join(removed_state_cols),
        })
        del trained, pred, part
        if variant["state_mode"] == "placebo":
            del variant_train_df, variant_test_df, placebo_part
        gc.collect()
    del train_df, test_df
    gc.collect()

score_panel_df = pd.concat(score_parts, ignore_index=True) if score_parts else pd.DataFrame()
model_meta_df = pd.DataFrame(model_meta_rows)
placebo_audit_df = pd.concat(placebo_audit_parts, ignore_index=True) if placebo_audit_parts else pd.DataFrame()
feature_importance_df = pd.DataFrame(feature_importance_rows)
del score_parts
gc.collect()
if len(placebo_audit_df):
    test_audit = placebo_audit_df[placebo_audit_df["sample"] == "test"].copy()
    test_audit = pd.merge(test_audit, pd.DataFrame(FOLD_PLAN)[["fold_id", "train_end"]], on="fold_id", how="left")
    test_audit["train_end"] = pd.to_datetime(test_audit["train_end"])
    if int((test_audit["source_state_date"] > test_audit["train_end"]).sum()) > 0:
        raise AssertionError("placebo test state uses post-cutoff source date")
model_meta_df.to_csv(OUT_DIR / "v85_model_meta.csv", index=False)
placebo_audit_df.to_csv(OUT_DIR / "v85_placebo_date_audit.csv", index=False)
feature_importance_df.to_csv(OUT_DIR / "v85_feature_importance.csv", index=False)
feature_group_rows = []
for (variant_name, feature_name, feature_group), gdf in feature_importance_df.groupby(["variant", "feature", "feature_group"]):
    feature_group_rows.append({
        "variant": variant_name, "feature": feature_name,
        "feature_group": feature_group,
        "importance_gain_mean": float(gdf["importance_gain"].mean()),
        "importance_split_mean": float(gdf["importance_split"].mean()),
    })
feature_importance_summary_df = pd.DataFrame(feature_group_rows)
feature_importance_summary_df.to_csv(OUT_DIR / "v85_feature_importance_summary.csv", index=False)

state_importance_rows = []
for (variant_name, fold_id), gdf in feature_importance_df.groupby(["variant", "fold_id"]):
    total_gain = float(gdf["importance_gain"].sum())
    state_gain = float(gdf.loc[gdf["feature_group"] == "market_state", "importance_gain"].sum())
    state_importance_rows.append({
        "variant": variant_name, "fold_id": fold_id, "total_gain": total_gain, "state_gain": state_gain,
        "state_gain_share": state_gain / total_gain if total_gain > 0 else np.nan,
    })
state_importance_df = pd.DataFrame(state_importance_rows)
state_importance_df.to_csv(OUT_DIR / "v85_state_importance_by_fold.csv", index=False)
print("score_panel:", score_panel_df.shape)
display_df(model_meta_df, 30)
display_df(state_importance_df, 30)
if len(score_panel_df) == 0:
    raise ValueError("empty OOS score panel")


## 7. 计算月度综合指标、分桶表现与 pooled ROC/PR 曲线


In [ ]:
monthly_rows = []
bucket_rows = []
enriched_parts = []
selected_rows = []
grouped = score_panel_df.groupby(["variant", "fold_id", DATE_COL])
for (variant, fold_id, dt), gdf in progress_iter(grouped, total=grouped.ngroups, desc="monthly diagnostics"):
    m = gdf.dropna(subset=["score", TARGET_COL]).copy()
    if len(m) < 50:
        continue
    m[STOCK_COL] = m[STOCK_COL].astype(str)
    m = m.sort_values("score", ascending=False).reset_index(drop=True)
    pred_order = m[STOCK_COL].tolist()
    true_order = m.sort_values(TARGET_COL, ascending=False)[STOCK_COL].tolist()
    gain_map = dict(zip(m[STOCK_COL], pd.to_numeric(m[TARGET_COL], errors="coerce")))
    true8 = set(true_order[:8])
    true20 = set(true_order[:20])
    raw8 = pred_order[:8]
    board8 = select_top8_board_cap(m)
    raw8_set = set(raw8)
    board8_set = set(board8)
    y8 = np.asarray([1 if s in true8 else 0 for s in m[STOCK_COL]], dtype=int)
    y20 = np.asarray([1 if s in true20 else 0 for s in m[STOCK_COL]], dtype=int)
    scores = np.asarray(m["score"], dtype=float)
    universe_mean = float(pd.to_numeric(m[TARGET_COL], errors="coerce").mean())
    raw8_alpha = float(np.nanmean([gain_map.get(s, np.nan) for s in raw8]))
    board8_alpha = float(np.nanmean([gain_map.get(s, np.nan) for s in board8]))
    bottom8 = pred_order[-8:]
    bottom8_alpha = float(np.nanmean([gain_map.get(s, np.nan) for s in bottom8]))
    row = {
        "variant": variant, "fold_id": fold_id, DATE_COL: pd.Timestamp(dt), "n_universe": int(len(m)),
        "rank_ic": safe_rank_ic(scores, m[TARGET_COL]),
        "pearson_ic": safe_pearson_ic(scores, m[TARGET_COL]),
        "roc_auc_true_top8": roc_auc_binary(y8, scores),
        "pr_auc_true_top8": average_precision_binary(y8, scores),
        "roc_auc_true_top20": roc_auc_binary(y20, scores),
        "pr_auc_true_top20": average_precision_binary(y20, scores),
        "precision_true_top8_at8_raw": len(raw8_set & true8) / 8.0,
        "recall_true_top8_at8_raw": len(raw8_set & true8) / 8.0,
        "precision_true_top20_at8_raw": len(raw8_set & true20) / 8.0,
        "recall_true_top20_at8_raw": len(raw8_set & true20) / 20.0,
        "precision_true_top20_at8_boardcap": len(board8_set & true20) / float(_bi.max(1, len(board8))),
        "recall_true_top20_at8_boardcap": len(board8_set & true20) / 20.0,
        "map_true_top20_at8": average_precision_at_k(pred_order, true20, 8),
        "ndcg_alpha_at8": ndcg_at_k(pred_order, gain_map, 8),
        "ndcg_alpha_at20": ndcg_at_k(pred_order, gain_map, 20),
        "prevalence_true_top8": 8.0 / len(m),
        "prevalence_true_top20": 20.0 / len(m),
        "precision_lift_top20_at8": (len(raw8_set & true20) / 8.0) / (20.0 / len(m)),
        "universe_alpha": universe_mean,
        "raw_top8_alpha": raw8_alpha,
        "boardcap_top8_alpha": board8_alpha,
        "boardcap_top8_edge": board8_alpha - universe_mean,
        "top_bottom8_spread": raw8_alpha - bottom8_alpha,
        "selected_count": int(len(board8)),
    }
    monthly_rows.append(row)

    score_map = dict(zip(m[STOCK_COL], pd.to_numeric(m["score"], errors="coerce")))
    for selected_rank, stock in enumerate(board8, 1):
        selected_rows.append({
            "variant": variant, "fold_id": fold_id, DATE_COL: pd.Timestamp(dt),
            "selected_rank": int(selected_rank), STOCK_COL: stock,
            "score": score_map.get(stock, np.nan),
            "realized_alpha": gain_map.get(stock, np.nan),
            "board": board_name(stock),
        })

    n = len(m)
    m["score_rank_pct"] = (np.arange(n, dtype=float) + 1.0) / float(n)
    m["true_top8"] = y8
    m["true_top20"] = y20
    enriched_parts.append(m[["variant", "fold_id", DATE_COL, STOCK_COL, "score_rank_pct", "true_top8", "true_top20"]])

    for bucket in range(1, BUCKET_N + 1):
        lo = int(math.floor((bucket - 1) * n / float(BUCKET_N)))
        hi = int(math.floor(bucket * n / float(BUCKET_N)))
        b = m.iloc[lo:hi]
        bucket_rows.append({
            "variant": variant, "fold_id": fold_id, DATE_COL: pd.Timestamp(dt),
            "score_bucket": bucket, "n": int(len(b)),
            "mean_alpha": float(pd.to_numeric(b[TARGET_COL], errors="coerce").mean()),
        })

monthly_metrics_df = pd.DataFrame(monthly_rows)
bucket_monthly_df = pd.DataFrame(bucket_rows)
curve_panel_df = pd.concat(enriched_parts, ignore_index=True) if enriched_parts else pd.DataFrame()
selected_detail_df = pd.DataFrame(selected_rows)
monthly_metrics_df.to_csv(OUT_DIR / "v85_monthly_metrics.csv", index=False)
bucket_monthly_df.to_csv(OUT_DIR / "v85_bucket_monthly.csv", index=False)
selected_detail_df.to_csv(OUT_DIR / "v85_selected_detail.csv", index=False)
print("monthly metrics:", monthly_metrics_df.shape, "bucket:", bucket_monthly_df.shape)
display_df(monthly_metrics_df, 20)


## 8. 汇总、fold 稳定性、相对 V46 基线和 bootstrap


In [ ]:
def summarize_variant(gdf):
    ret = pd.to_numeric(gdf["boardcap_top8_edge"], errors="coerce").dropna()
    return {
        "months": int(len(gdf)),
        "rank_ic_mean": float(gdf["rank_ic"].mean()),
        "rank_ic_median": float(gdf["rank_ic"].median()),
        "rank_ic_positive_rate": float((gdf["rank_ic"] > 0).mean()),
        "rank_ic_ir_annualized": icir(gdf["rank_ic"]),
        "roc_auc_true_top20_mean": float(gdf["roc_auc_true_top20"].mean()),
        "pr_auc_true_top20_mean": float(gdf["pr_auc_true_top20"].mean()),
        "precision_true_top20_at8_mean": float(gdf["precision_true_top20_at8_raw"].mean()),
        "recall_true_top20_at8_mean": float(gdf["recall_true_top20_at8_raw"].mean()),
        "map_true_top20_at8_mean": float(gdf["map_true_top20_at8"].mean()),
        "ndcg_alpha_at8_mean": float(gdf["ndcg_alpha_at8"].mean()),
        "ndcg_alpha_at20_mean": float(gdf["ndcg_alpha_at20"].mean()),
        "precision_lift_top20_at8_mean": float(gdf["precision_lift_top20_at8"].mean()),
        "boardcap_top8_alpha_mean": float(gdf["boardcap_top8_alpha"].mean()),
        "boardcap_top8_edge_mean": float(gdf["boardcap_top8_edge"].mean()),
        "boardcap_top8_edge_annualized": annualized_return(ret),
        "boardcap_top8_edge_cumulative": float((1.0 + ret).prod() - 1.0) if len(ret) else np.nan,
        "boardcap_top8_edge_mdd": calc_mdd(ret),
        "boardcap_top8_edge_win_rate": float((ret > 0).mean()) if len(ret) else np.nan,
        "boardcap_top8_edge_worst_month": float(ret.min()) if len(ret) else np.nan,
        "top_bottom8_spread_mean": float(gdf["top_bottom8_spread"].mean()),
    }


summary_rows = []
for variant, gdf in monthly_metrics_df.groupby("variant"):
    row = {"variant": variant}
    row.update(summarize_variant(gdf.sort_values(DATE_COL)))
    summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows)

fold_rows = []
for (variant, fold_id), gdf in monthly_metrics_df.groupby(["variant", "fold_id"]):
    row = {"variant": variant, "fold_id": fold_id}
    row.update(summarize_variant(gdf.sort_values(DATE_COL)))
    fold_rows.append(row)
fold_summary_df = pd.DataFrame(fold_rows)

baseline = monthly_metrics_df[monthly_metrics_df["variant"] == "A0_full_v46"].copy()
baseline_cols = [DATE_COL, "fold_id", "rank_ic", "pr_auc_true_top20", "precision_true_top20_at8_raw", "ndcg_alpha_at8", "boardcap_top8_edge"]
baseline = baseline[baseline_cols].copy()
baseline = baseline.rename(columns=dict((c, c + "_a0") for c in baseline_cols if c not in [DATE_COL, "fold_id"]))

compare_rows = []
for variant, gdf in monthly_metrics_df.groupby("variant"):
    joined = pd.merge(gdf, baseline, on=[DATE_COL, "fold_id"], how="inner")
    row = {"variant": variant, "paired_months": int(len(joined))}
    for metric in ["rank_ic", "pr_auc_true_top20", "precision_true_top20_at8_raw", "ndcg_alpha_at8", "boardcap_top8_edge"]:
        delta = pd.to_numeric(joined[metric], errors="coerce") - pd.to_numeric(joined[metric + "_a0"], errors="coerce")
        row[metric + "_delta_vs_a0"] = float(delta.mean())
        row[metric + "_win_rate_vs_a0"] = float((delta > 0).mean())
        if metric in ["rank_ic", "boardcap_top8_edge"]:
            lo, hi, ppos = moving_block_bootstrap_mean(delta)
            row[metric + "_delta_boot_ci_low"] = lo
            row[metric + "_delta_boot_ci_high"] = hi
            row[metric + "_delta_boot_prob_positive"] = ppos
    compare_rows.append(row)
comparison_df = pd.DataFrame(compare_rows)

def compare_pair(challenger, control):
    left = monthly_metrics_df[monthly_metrics_df["variant"] == challenger].copy()
    right = monthly_metrics_df[monthly_metrics_df["variant"] == control].copy()
    metrics = ["rank_ic", "pr_auc_true_top20", "precision_true_top20_at8_raw", "ndcg_alpha_at8", "boardcap_top8_edge"]
    right = right[[DATE_COL, "fold_id"] + metrics].rename(columns=dict((c, c + "_control") for c in metrics))
    joined = pd.merge(left, right, on=[DATE_COL, "fold_id"], how="inner")
    row = {"challenger": challenger, "control": control, "paired_months": int(len(joined))}
    for metric in metrics:
        delta = pd.to_numeric(joined[metric], errors="coerce") - pd.to_numeric(joined[metric + "_control"], errors="coerce")
        row[metric + "_delta"] = float(delta.mean())
        row[metric + "_win_rate"] = float((delta > 0).mean())
        if metric in ["rank_ic", "boardcap_top8_edge"]:
            lo, hi, ppos = moving_block_bootstrap_mean(delta)
            row[metric + "_boot_ci_low"] = lo
            row[metric + "_boot_ci_high"] = hi
            row[metric + "_boot_prob_positive"] = ppos
    return row


pairwise_rows = []
for challenger, control in [
    ("M1_true_state", "A0_full_v46"),
    ("M1_true_state", "P1_placebo_state"),
    ("P1_placebo_state", "A0_full_v46"),
]:
    if challenger in set(monthly_metrics_df["variant"]) and control in set(monthly_metrics_df["variant"]):
        pairwise_rows.append(compare_pair(challenger, control))
pairwise_comparison_df = pd.DataFrame(pairwise_rows)

period_rows = []
period_specs = [
    ("weak_2022_2023", 2022, 2023),
    ("middle_2024", 2024, 2024),
    ("strong_2025_2026", 2025, 2026),
]
for period_name, year_start, year_end in period_specs:
    part = monthly_metrics_df[(monthly_metrics_df[DATE_COL].dt.year >= year_start) & (monthly_metrics_df[DATE_COL].dt.year <= year_end)]
    for variant, gdf in part.groupby("variant"):
        row = {"period": period_name, "variant": variant}
        row.update(summarize_variant(gdf.sort_values(DATE_COL)))
        period_rows.append(row)
period_summary_df = pd.DataFrame(period_rows)

yearly_rows = []
for (variant, year), gdf in monthly_metrics_df.groupby(["variant", monthly_metrics_df[DATE_COL].dt.year]):
    row = {"variant": variant, "year": int(year)}
    row.update(summarize_variant(gdf.sort_values(DATE_COL)))
    yearly_rows.append(row)
yearly_summary_df = pd.DataFrame(yearly_rows)

# Posterior diagnostic only: locate state regions where M1 helps or hurts.
state_month_map = pd.merge(
    date_map, market_state_df[["feature_date"] + STATE_FEATURE_COLS],
    on="feature_date", how="left",
).drop_duplicates(DATE_COL)
state_condition_rows = []
for control in ["A0_full_v46", "P1_placebo_state"]:
    m1_part = monthly_metrics_df[monthly_metrics_df["variant"] == "M1_true_state"][[DATE_COL, "fold_id", "boardcap_top8_edge", "rank_ic"]].copy()
    control_part = monthly_metrics_df[monthly_metrics_df["variant"] == control][[DATE_COL, "fold_id", "boardcap_top8_edge", "rank_ic"]].copy()
    control_part = control_part.rename(columns={"boardcap_top8_edge": "control_edge", "rank_ic": "control_rank_ic"})
    joined = pd.merge(m1_part, control_part, on=[DATE_COL, "fold_id"], how="inner")
    joined = pd.merge(joined, state_month_map, on=DATE_COL, how="left")
    joined["edge_delta"] = joined["boardcap_top8_edge"] - joined["control_edge"]
    joined["rank_ic_delta"] = joined["rank_ic"] - joined["control_rank_ic"]
    for state_col in STATE_FEATURE_COLS:
        valid = joined.dropna(subset=[state_col, "edge_delta"]).copy()
        n = len(valid)
        if n < 9:
            continue
        ranks = valid[state_col].rank(method="first", ascending=True)
        valid["state_tercile"] = np.floor((ranks.values - 1.0) * 3.0 / float(n)).astype(int) + 1
        valid["state_tercile"] = valid["state_tercile"].clip(lower=1, upper=3)
        for tercile, gdf in valid.groupby("state_tercile"):
            state_condition_rows.append({
                "control": control, "state_feature": state_col, "state_tercile": int(tercile),
                "months": int(len(gdf)), "state_value_min": float(gdf[state_col].min()),
                "state_value_max": float(gdf[state_col].max()), "state_value_mean": float(gdf[state_col].mean()),
                "edge_delta_mean": float(gdf["edge_delta"].mean()),
                "edge_delta_win_rate": float((gdf["edge_delta"] > 0).mean()),
                "rank_ic_delta_mean": float(gdf["rank_ic_delta"].mean()),
            })
state_conditioned_uplift_df = pd.DataFrame(state_condition_rows)

a0_fold = fold_summary_df[fold_summary_df["variant"] == "A0_full_v46"][["fold_id", "boardcap_top8_edge_mean", "rank_ic_mean"]].copy()
a0_fold = a0_fold.rename(columns={"boardcap_top8_edge_mean": "boardcap_top8_edge_mean_a0", "rank_ic_mean": "rank_ic_mean_a0"})
fold_delta_df = pd.merge(fold_summary_df, a0_fold, on="fold_id", how="left")
fold_delta_df["top8_edge_delta_vs_a0"] = fold_delta_df["boardcap_top8_edge_mean"] - fold_delta_df["boardcap_top8_edge_mean_a0"]
fold_delta_df["rank_ic_delta_vs_a0"] = fold_delta_df["rank_ic_mean"] - fold_delta_df["rank_ic_mean_a0"]

summary_df.to_csv(OUT_DIR / "v85_variant_summary.csv", index=False)
fold_summary_df.to_csv(OUT_DIR / "v85_fold_summary.csv", index=False)
comparison_df.to_csv(OUT_DIR / "v85_comparison_vs_a0.csv", index=False)
pairwise_comparison_df.to_csv(OUT_DIR / "v85_pairwise_comparison.csv", index=False)
period_summary_df.to_csv(OUT_DIR / "v85_period_summary.csv", index=False)
yearly_summary_df.to_csv(OUT_DIR / "v85_yearly_summary.csv", index=False)
state_conditioned_uplift_df.to_csv(OUT_DIR / "v85_state_conditioned_uplift.csv", index=False)
fold_delta_df.to_csv(OUT_DIR / "v85_fold_delta_vs_a0.csv", index=False)
display_df(summary_df, 20)
display_df(comparison_df, 20)
display_df(pairwise_comparison_df, 20)
display_df(period_summary_df, 20)
display_df(state_conditioned_uplift_df, 30)


## 9. 持仓变化、失败月份与预注册判定


In [ ]:
# Top8 holding overlap versus exact V46 baseline.
selected_sets = {}
for (variant, fold_id, dt), gdf in selected_detail_df.groupby(["variant", "fold_id", DATE_COL]):
    selected_sets[(variant, fold_id, pd.Timestamp(dt))] = set(gdf[STOCK_COL].astype(str).tolist())
overlap_rows = []
for key, challenger_set in selected_sets.items():
    variant, fold_id, dt = key
    a0_set = selected_sets.get(("A0_full_v46", fold_id, dt), set())
    union = challenger_set | a0_set
    overlap_rows.append({
        "variant": variant, "fold_id": fold_id, DATE_COL: dt,
        "overlap_count_vs_a0": int(len(challenger_set & a0_set)),
        "jaccard_vs_a0": float(len(challenger_set & a0_set)) / float(len(union)) if len(union) else np.nan,
    })
holding_overlap_monthly_df = pd.DataFrame(overlap_rows)
holding_overlap_summary_rows = []
for variant, gdf in holding_overlap_monthly_df.groupby("variant"):
    holding_overlap_summary_rows.append({
        "variant": variant, "months": int(len(gdf)),
        "overlap_count_vs_a0_mean": float(gdf["overlap_count_vs_a0"].mean()),
        "jaccard_vs_a0_mean": float(gdf["jaccard_vs_a0"].mean()),
    })
holding_overlap_summary_df = pd.DataFrame(holding_overlap_summary_rows)

# Diagnose whether a variant repairs A0 failure months or creates new failures.
failure_rows = []
a0_monthly = monthly_metrics_df[monthly_metrics_df["variant"] == "A0_full_v46"][["fold_id", DATE_COL, "boardcap_top8_edge"]].copy()
a0_monthly = a0_monthly.rename(columns={"boardcap_top8_edge": "a0_top8_edge"})
for variant, gdf in monthly_metrics_df.groupby("variant"):
    joined = pd.merge(gdf[["fold_id", DATE_COL, "boardcap_top8_edge"]], a0_monthly, on=["fold_id", DATE_COL], how="inner")
    for _, row in joined.iterrows():
        challenger_edge = float(row["boardcap_top8_edge"])
        a0_edge = float(row["a0_top8_edge"])
        if a0_edge <= 0 and challenger_edge > 0:
            state = "repair_a0_failure"
        elif a0_edge > 0 and challenger_edge <= 0:
            state = "create_new_failure"
        elif a0_edge <= 0 and challenger_edge <= 0:
            state = "both_fail"
        else:
            state = "both_positive"
        failure_rows.append({
            "variant": variant, "fold_id": row["fold_id"], DATE_COL: pd.Timestamp(row[DATE_COL]),
            "a0_top8_edge": a0_edge, "variant_top8_edge": challenger_edge,
            "delta_vs_a0": challenger_edge - a0_edge, "failure_state": state,
        })
failure_month_tradeoff_df = pd.DataFrame(failure_rows)
failure_summary_rows = []
for variant, gdf in failure_month_tradeoff_df.groupby("variant"):
    failure_summary_rows.append({
        "variant": variant, "months": int(len(gdf)),
        "repair_a0_failure_count": int((gdf["failure_state"] == "repair_a0_failure").sum()),
        "create_new_failure_count": int((gdf["failure_state"] == "create_new_failure").sum()),
        "both_fail_count": int((gdf["failure_state"] == "both_fail").sum()),
        "both_positive_count": int((gdf["failure_state"] == "both_positive").sum()),
    })
failure_summary_df = pd.DataFrame(failure_summary_rows)

holding_overlap_monthly_df.to_csv(OUT_DIR / "v85_holding_overlap_monthly.csv", index=False)
holding_overlap_summary_df.to_csv(OUT_DIR / "v85_holding_overlap_summary.csv", index=False)
failure_month_tradeoff_df.to_csv(OUT_DIR / "v85_failure_month_tradeoff.csv", index=False)
failure_summary_df.to_csv(OUT_DIR / "v85_failure_summary.csv", index=False)

# Pre-registered rules. M1 must beat both exact V46 and temporally invalid placebo.
decision_rows = [
    {"variant": "A0_full_v46", "decision": "baseline"},
    {"variant": "P1_placebo_state", "decision": "negative_control"},
]
m1_a0 = pairwise_comparison_df[
    (pairwise_comparison_df["challenger"] == "M1_true_state") &
    (pairwise_comparison_df["control"] == "A0_full_v46")
]
m1_p1 = pairwise_comparison_df[
    (pairwise_comparison_df["challenger"] == "M1_true_state") &
    (pairwise_comparison_df["control"] == "P1_placebo_state")
]
m1_folds = fold_delta_df[fold_delta_df["variant"] == "M1_true_state"]
if len(m1_a0) and len(m1_p1) and len(m1_folds):
    a0_row = m1_a0.iloc[0]
    p1_row = m1_p1.iloc[0]
    positive_folds = int((m1_folds["top8_edge_delta_vs_a0"] > 0).sum())
    worst_fold_delta = float(m1_folds["top8_edge_delta_vs_a0"].min())
    weak = period_summary_df[period_summary_df["period"] == "weak_2022_2023"]
    strong = period_summary_df[period_summary_df["period"] == "strong_2025_2026"]
    weak_m1 = weak[weak["variant"] == "M1_true_state"]
    weak_a0 = weak[weak["variant"] == "A0_full_v46"]
    strong_m1 = strong[strong["variant"] == "M1_true_state"]
    strong_a0 = strong[strong["variant"] == "A0_full_v46"]
    weak_delta = float(weak_m1["boardcap_top8_edge_mean"].iloc[0] - weak_a0["boardcap_top8_edge_mean"].iloc[0]) if len(weak_m1) and len(weak_a0) else np.nan
    strong_delta = float(strong_m1["boardcap_top8_edge_mean"].iloc[0] - strong_a0["boardcap_top8_edge_mean"].iloc[0]) if len(strong_m1) and len(strong_a0) else np.nan
    overlap = holding_overlap_summary_df[holding_overlap_summary_df["variant"] == "M1_true_state"]
    overlap_mean = float(overlap["overlap_count_vs_a0_mean"].iloc[0]) if len(overlap) else np.nan
    gates = {
        "edge_delta_vs_a0_positive": float(a0_row["boardcap_top8_edge_delta"]) > 0,
        "edge_bootstrap_ci_low_vs_a0_positive": float(a0_row["boardcap_top8_edge_boot_ci_low"]) > 0,
        "edge_delta_vs_placebo_positive": float(p1_row["boardcap_top8_edge_delta"]) > 0,
        "edge_bootstrap_prob_vs_placebo_ge_80pct": float(p1_row["boardcap_top8_edge_boot_prob_positive"]) >= 0.80,
        "positive_fold_count_ge_3": positive_folds >= 3,
        "worst_fold_not_worse_20bp": worst_fold_delta >= -0.002,
        "weak_2022_2023_improves": weak_delta > 0,
        "strong_2025_2026_damage_within_20bp": strong_delta >= -0.002,
        "rank_ic_not_worse_20bp": float(a0_row["rank_ic_delta"]) >= -0.002,
        "precision_not_worse_1pct": float(a0_row["precision_true_top20_at8_raw_delta"]) >= -0.01,
        "mean_top8_overlap_ge_4": overlap_mean >= 4.0,
    }
    if _bi.all(gates.values()):
        decision = "jq_backtest_candidate"
    elif float(a0_row["boardcap_top8_edge_delta"]) > 0 and positive_folds >= 3 and float(a0_row["boardcap_top8_edge_boot_prob_positive"]) >= 0.80:
        decision = "watch"
    else:
        decision = "reject"
    out = {
        "variant": "M1_true_state", "decision": decision,
        "mean_top8_edge_delta_vs_a0": float(a0_row["boardcap_top8_edge_delta"]),
        "mean_top8_edge_delta_vs_placebo": float(p1_row["boardcap_top8_edge_delta"]),
        "rank_ic_delta_vs_a0": float(a0_row["rank_ic_delta"]),
        "precision_delta_vs_a0": float(a0_row["precision_true_top20_at8_raw_delta"]),
        "positive_fold_count": positive_folds, "available_fold_count": int(len(m1_folds)),
        "worst_fold_delta_vs_a0": worst_fold_delta,
        "weak_2022_2023_delta_vs_a0": weak_delta,
        "strong_2025_2026_delta_vs_a0": strong_delta,
        "mean_top8_overlap_vs_a0": overlap_mean,
        "bootstrap_ci_low_vs_a0": float(a0_row["boardcap_top8_edge_boot_ci_low"]),
        "bootstrap_prob_positive_vs_a0": float(a0_row["boardcap_top8_edge_boot_prob_positive"]),
        "bootstrap_prob_positive_vs_placebo": float(p1_row["boardcap_top8_edge_boot_prob_positive"]),
    }
    out.update(gates)
    decision_rows.append(out)
decision_df = pd.DataFrame(decision_rows)
decision_df.to_csv(OUT_DIR / "v85_pre_registered_decision_table.csv", index=False)
display_df(holding_overlap_summary_df, 20)
display_df(failure_summary_df, 20)
display_df(decision_df, 20)


## 10. 可视化仪表板


In [ ]:
variants = [v for v in RUN_VARIANTS if v in set(summary_df["variant"])]
x = np.arange(len(variants))

# 1) Core metric dashboard.
fig, axes = plt.subplots(2, 4, figsize=(19, 8))
dashboard_metrics = [
    ("rank_ic_mean", "Mean RankIC", 0.0),
    ("roc_auc_true_top20_mean", "ROC-AUC: true Top20", 0.5),
    ("pr_auc_true_top20_mean", "PR-AUC: true Top20", None),
    ("precision_true_top20_at8_mean", "Precision@8: true Top20", None),
    ("recall_true_top20_at8_mean", "Recall@8: true Top20", None),
    ("map_true_top20_at8_mean", "MAP@8: true Top20", None),
    ("ndcg_alpha_at8_mean", "NDCG@8", None),
    ("boardcap_top8_edge_mean", "Monthly Top8 edge", 0.0),
]
for ax, item in zip(axes.ravel(), dashboard_metrics):
    col, title, ref = item
    vals = []
    for v in variants:
        vals.append(float(summary_df.loc[summary_df["variant"] == v, col].iloc[0]))
    ax.bar(x, vals, color=[COLORS.get(v, "#777777") for v in variants])
    if ref is not None:
        ax.axhline(ref, color="#555555", linestyle="--", linewidth=1)
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels([v.split("_")[0] for v in variants])
    ax.grid(axis="y", alpha=0.25)
save_show(fig, "v85_core_metric_dashboard.png")

# 2) Pooled ROC and PR curves; scores are normalized to monthly rank percentiles first.
curve_rows = []
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for v in variants:
    d = curve_panel_df[curve_panel_df["variant"] == v]
    curve = binary_curve_points(d["true_top20"], -pd.to_numeric(d["score_rank_pct"], errors="coerce"))
    if len(curve) == 0:
        continue
    auc = roc_auc_binary(d["true_top20"], -pd.to_numeric(d["score_rank_pct"], errors="coerce"))
    ap = average_precision_binary(d["true_top20"], -pd.to_numeric(d["score_rank_pct"], errors="coerce"))
    axes[0].plot(curve["fpr"], curve["tpr"], color=COLORS.get(v), label="%s AUC=%.3f" % (v.split("_")[0], auc))
    axes[1].plot(curve["recall"], curve["precision"], color=COLORS.get(v), label="%s AP=%.3f" % (v.split("_")[0], ap))
    sampled = curve.iloc[::_bi.max(1, int(len(curve) / 200))].copy()
    sampled["variant"] = v
    sampled["pooled_auc"] = auc
    sampled["pooled_ap"] = ap
    curve_rows.append(sampled)
prevalence = float(curve_panel_df["true_top20"].mean())
axes[0].plot([0, 1], [0, 1], "--", color="#777777", label="random")
axes[0].set_title("Pooled ROC: true future Top20")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[1].axhline(prevalence, linestyle="--", color="#777777", label="random prevalence")
axes[1].set_title("Pooled Precision-Recall: true future Top20")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
save_show(fig, "v85_pooled_roc_pr_curves.png")
curve_points_df = pd.concat(curve_rows, ignore_index=True) if curve_rows else pd.DataFrame()
curve_points_df.to_csv(OUT_DIR / "v85_pooled_curve_points.csv", index=False)

# 3) Cumulative realized Top8 edge and rolling RankIC.
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
for v in variants:
    d = monthly_metrics_df[monthly_metrics_df["variant"] == v].sort_values(DATE_COL)
    axes[0].plot(d[DATE_COL], calc_nav(d["boardcap_top8_edge"]).values, color=COLORS.get(v), label=v.split("_")[0])
    axes[1].plot(d[DATE_COL], d["rank_ic"].rolling(6, min_periods=3).mean(), color=COLORS.get(v), label=v.split("_")[0])
axes[0].set_title("Cumulative Top8 edge (proxy)")
axes[0].set_ylabel("NAV")
axes[1].set_title("Rolling 6-month RankIC")
axes[1].axhline(0, color="#555555", linewidth=1)
axes[1].set_ylabel("RankIC")
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend(ncol=len(variants), fontsize=8)
save_show(fig, "v85_oos_nav_and_rankic.png")

# 4) Distribution stability.
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
rank_data = [monthly_metrics_df.loc[monthly_metrics_df["variant"] == v, "rank_ic"].dropna().values for v in variants]
edge_data = [monthly_metrics_df.loc[monthly_metrics_df["variant"] == v, "boardcap_top8_edge"].dropna().values for v in variants]
axes[0].boxplot(rank_data, labels=[v.split("_")[0] for v in variants], showmeans=True)
axes[1].boxplot(edge_data, labels=[v.split("_")[0] for v in variants], showmeans=True)
axes[0].set_title("Monthly RankIC distribution")
axes[1].set_title("Monthly Top8 edge distribution")
for ax in axes:
    ax.axhline(0, color="#777777", linestyle="--", linewidth=1)
    ax.grid(axis="y", alpha=0.25)
save_show(fig, "v85_metric_distributions.png")

# 5) Score bucket monotonicity. Bucket 1 is highest predicted score.
bucket_summary_df = bucket_monthly_df.groupby(["variant", "score_bucket"])["mean_alpha"].mean().reset_index()
bucket_summary_df.to_csv(OUT_DIR / "v85_bucket_summary.csv", index=False)
fig, ax = plt.subplots(figsize=(11, 5.5))
for v in variants:
    d = bucket_summary_df[bucket_summary_df["variant"] == v].sort_values("score_bucket")
    ax.plot(d["score_bucket"], d["mean_alpha"], marker="o", color=COLORS.get(v), label=v.split("_")[0])
ax.axhline(0, color="#777777", linestyle="--", linewidth=1)
ax.set_xticks(range(1, BUCKET_N + 1))
ax.set_title("Realized alpha by predicted score bucket (1 = highest)")
ax.set_xlabel("Score bucket")
ax.set_ylabel("Mean monthly alpha")
ax.grid(alpha=0.25)
ax.legend(fontsize=8)
save_show(fig, "v85_score_bucket_profile.png")

# 6) Fold heatmap for realized Top8 edge.
fold_ids = [x["fold_id"] for x in FOLD_PLAN if x["fold_id"] in set(fold_summary_df["fold_id"])]
heat = np.full((len(variants), len(fold_ids)), np.nan)
for i, v in enumerate(variants):
    for j, f in enumerate(fold_ids):
        d = fold_summary_df[(fold_summary_df["variant"] == v) & (fold_summary_df["fold_id"] == f)]
        if len(d):
            heat[i, j] = float(d["boardcap_top8_edge_mean"].iloc[0])
fig, ax = plt.subplots(figsize=(12, 4.8))
im = ax.imshow(heat, aspect="auto", cmap="RdYlGn")
ax.set_xticks(np.arange(len(fold_ids)))
ax.set_xticklabels(fold_ids, rotation=30, ha="right")
ax.set_yticks(np.arange(len(variants)))
ax.set_yticklabels([v.split("_")[0] for v in variants])
ax.set_title("Mean monthly Top8 edge by walk-forward fold")
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        if np.isfinite(heat[i, j]):
            ax.text(j, i, "%.3f" % heat[i, j], ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.8)
save_show(fig, "v85_fold_stability_heatmap.png")

# 7) Market-state timeline and state importance share.
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
timeline_cols = [
    "state_csi800_ret60", "state_csi800_vol20",
    "state_csi800_above_ma60", "state_csi800_ret20_dispersion",
]
for col in timeline_cols:
    s = pd.to_numeric(market_state_df[col], errors="coerce")
    z = (s - s.mean()) / s.std() if float(s.std()) > 1e-12 else s * np.nan
    axes[0].plot(market_state_df["feature_date"], z, label=col.replace("state_csi800_", ""))
axes[0].axhline(0, color="#555555", linewidth=1)
axes[0].set_title("CSI800 market-state indicators (z-score)")
axes[0].set_ylabel("z-score")
axes[0].legend(fontsize=8, ncol=2)
importance_plot = state_importance_df.groupby("variant")["state_gain_share"].mean().reindex(variants)
axes[1].bar(np.arange(len(importance_plot)), importance_plot.values, color=[COLORS.get(v, "#777777") for v in importance_plot.index])
axes[1].set_xticks(np.arange(len(importance_plot)))
axes[1].set_xticklabels([v.split("_")[0] for v in importance_plot.index])
axes[1].set_title("Mean LightGBM gain share from state features")
axes[1].set_ylabel("gain share")
for ax in axes:
    ax.grid(axis="y", alpha=0.25)
save_show(fig, "v85_state_timeline_and_importance_share.png")

state_feature_plot = feature_importance_summary_df[
    feature_importance_summary_df["feature_group"] == "market_state"
].copy()
state_feature_plot = state_feature_plot.sort_values(["variant", "importance_gain_mean"], ascending=[True, False])
state_feature_plot.to_csv(OUT_DIR / "v85_state_feature_importance.csv", index=False)
fig, ax = plt.subplots(figsize=(12, 6))
plot_features = _bi.sorted(set(state_feature_plot["feature"].tolist()))
y = np.arange(len(plot_features))
width = 0.36
for offset, variant in zip([-width / 2.0, width / 2.0], ["M1_true_state", "P1_placebo_state"]):
    d = state_feature_plot[state_feature_plot["variant"] == variant].set_index("feature").reindex(plot_features)
    ax.barh(y + offset, d["importance_gain_mean"].fillna(0).values, height=width, color=COLORS.get(variant), label=variant.split("_")[0])
ax.set_yticks(y)
ax.set_yticklabels([x.replace("state_", "") for x in plot_features], fontsize=8)
ax.set_title("True-state versus placebo-state feature importance")
ax.set_xlabel("mean gain")
ax.grid(axis="x", alpha=0.25)
ax.legend(fontsize=8)
save_show(fig, "v85_state_feature_importance.png")

# 8) Economic delta and Top8 overlap versus A0.
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
challengers = [v for v in variants if v != "A0_full_v46"]
for v in challengers:
    d = failure_month_tradeoff_df[failure_month_tradeoff_df["variant"] == v].sort_values(DATE_COL)
    axes[0].plot(d[DATE_COL], d["delta_vs_a0"].rolling(6, min_periods=3).mean(), color=COLORS.get(v), label=v.split("_")[0])
axes[0].axhline(0, color="#555555", linewidth=1)
axes[0].set_title("Rolling 6-month Top8 edge delta vs A0")
overlap_plot = holding_overlap_summary_df.set_index("variant").reindex(challengers)
axes[1].bar(np.arange(len(challengers)), overlap_plot["overlap_count_vs_a0_mean"].values, color=[COLORS.get(v) for v in challengers])
axes[1].set_xticks(np.arange(len(challengers)))
axes[1].set_xticklabels([v.split("_")[0] for v in challengers])
axes[1].set_ylim(0, STOCK_NUM)
axes[1].set_title("Mean Top8 holding overlap versus A0")
for ax in axes:
    ax.grid(axis="y", alpha=0.25)
    ax.legend(fontsize=8) if ax is axes[0] else None
save_show(fig, "v85_delta_and_holding_overlap.png")

# 9) Posterior state-conditional uplift. This explains results; it is not a tuning rule.
condition_features = [
    "state_csi800_ret60", "state_csi800_vol20",
    "state_csi800_above_ma60", "state_csi800_ret20_dispersion",
]
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, state_col in zip(axes.ravel(), condition_features):
    d = state_conditioned_uplift_df[
        (state_conditioned_uplift_df["control"] == "A0_full_v46") &
        (state_conditioned_uplift_df["state_feature"] == state_col)
    ].sort_values("state_tercile")
    ax.bar(d["state_tercile"].values, d["edge_delta_mean"].values, color="#00a087")
    ax.axhline(0, color="#555555", linewidth=1)
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(["low", "mid", "high"])
    ax.set_title(state_col.replace("state_csi800_", ""))
    ax.set_ylabel("M1 edge delta vs A0")
    ax.grid(axis="y", alpha=0.25)
save_show(fig, "v85_state_conditioned_uplift.png")


## 11. 结果文件与解读提醒


In [ ]:
readme = [
    "V85 market-state conditioning interpretation notes",
    "1. A0_full_v46 is the exact V46 control: raw alpha_1m, L2, fixed120.",
    "2. M1 changes only the feature matrix by adding point-in-time market-state columns.",
    "3. P1 has the same state dimension as M1, but training months are permuted and test states are sampled only from training-period months.",
    "4. ROC/PR/RankIC diagnose ranking; boardcap Top8 edge is the primary economic endpoint.",
    "5. M1 must beat both A0 and P1, improve at least three folds, preserve ranking health, and avoid material strong-window damage.",
    "6. The automatic decision only grants eligibility for an aligned JoinQuant backtest; it is not production approval.",
    "7. LABEL_BOUNDARY_MODE remains legacy_rebalance to reproduce V46 exactly; absolute OOS evidence must be read with the existing label-safe audits.",
    "8. State features are constant across stocks within one month; they condition tree splits but cannot rank stocks on their own.",
    "9. Set REBUILD_MARKET_STATE=True only when new feature dates must be fetched; otherwise the cache is reused.",
]
with open(OUT_DIR / "v85_README.txt", "w") as f:
    f.write("\n".join(readme))

print("saved outputs:")
for fp in _bi.sorted(OUT_DIR.glob("v85_*")):
    print("-", fp)
print("figures:")
for fp in _bi.sorted(FIG_DIR.glob("*.png")):
    print("-", fp)
print("\nKey reminder: market state is useful only if M1 beats both exact V46 and the temporal placebo.")
